In [49]:
import pickle

# Paths to your pickle files
merge_file = r"D:\MULTIMODAL-RAG\aic25_batch1_siglip2_taskformer.pkl"
florence2_file = r"D:\MULTIMODAL-RAG\aic25_batch1_florence2_ocr_padded.pkl"
yolo11_file = r"D:\MULTIMODAL-RAG\aic25_batch1_yolo11_objects.pkl"
colours_file = r"D:\MULTIMODAL-RAG\aic25_batch1_colours.pkl"

# Load pickles
with open(merge_file, "rb") as f:
    merge_data = pickle.load(f)
print(f"Loaded {len(merge_data)} items from {merge_file}")
# print("First item:", merge_data[0])

with open(florence2_file, "rb") as f:
    florence2_data = pickle.load(f)
print(f"Loaded {len(florence2_data)} items from {florence2_file}")
print("First item:", florence2_data[0])
    
with open(yolo11_file, "rb") as f:
    yolo11_data = pickle.load(f)
print(f"Loaded {len(yolo11_data)} items from {yolo11_file}")
print("First item:", yolo11_data[0])

with open(colours_file, "rb") as f:
    colours_data = pickle.load(f)
print(f"Loaded {len(colours_data)} items from {colours_file}")
print("First item:", colours_data[0])

# Sort both lists by path
merge_sorted = sorted(merge_data, key=lambda x: x["path"])
florence2_sorted = sorted(florence2_data, key=lambda x: x["path"])
yolo11_sorted = sorted(yolo11_data, key=lambda x: x["path"])
colours_sorted = sorted(colours_data, key=lambda x: x["path"])

# Check lengths first
assert len(merge_sorted) == len(florence2_sorted) == len(yolo11_sorted) == len(colours_sorted), "Different number of items!"

Loaded 177321 items from D:\MULTIMODAL-RAG\aic25_batch1_siglip2_taskformer.pkl
Loaded 177321 items from D:\MULTIMODAL-RAG\aic25_batch1_florence2_ocr_padded.pkl
First item: {'path': 'L21_V001/001.jpg', 'ocr': 'htv9<pad>hd<pad>06:30:11<pad>giay'}
Loaded 177321 items from D:\MULTIMODAL-RAG\aic25_batch1_yolo11_objects.pkl
First item: {'path': 'L21_V001/001.jpg', 'detections': [{'xywh': [[566.5096435546875, 363.19610595703125, 696.651611328125, 322.527587890625]], 'xywhn': [[0.44258567690849304, 0.5044390559196472, 0.5442590713500977, 0.4479549825191498]], 'xyxy': [[218.183837890625, 201.93231201171875, 914.83544921875, 524.4598999023438]], 'xyxyn': [[0.17045612633228302, 0.28046154975891113, 0.7147151827812195, 0.7284165620803833]], 'names': ['airplane'], 'confs': [0.32402315735816956]}]}
Loaded 177321 items from D:\MULTIMODAL-RAG\aic25_batch1_colours.pkl
First item: {'path': 'L21_V001/001.jpg', 'colours': [(144, 56, 42), (173, 65, 43), (118, 47, 35), (119, 58, 65), (207, 76, 36), (83, 43,

In [ ]:
merged = []
for i, (m, f, y, c) in enumerate(zip(merge_sorted, florence2_sorted, yolo11_sorted, colours_sorted)):
    assert m["path"] == f["path"] == y["path"] == c["path"], \
        f"Path mismatch at index {i}: {m['path']} != {f['path']} != {y['path']} != {c['path']}"
    
    ocr_text = f["ocr"].replace("<pad>", "<sep>").strip()
    if len(ocr_text) > 4096:
        ocr_text = ocr_text[:4096]

    merged.append({
        "path": m["path"],
        "text_dense": m["text_dense"],
        "sketch_dense": m["sketch_dense"],
        "ocr": ocr_text,
        "objects": [name for det in y["detections"] for name in det["names"]],
        "colours": c["main_colours"]
    })

# Sort final merged list by path
merged_sorted = sorted(merged, key=lambda x: x["path"])

In [55]:
# Add index column
# for i, item in enumerate(merged_sorted):
#     item["index"] = i

print(f"Final merged: {len(merged_sorted)} items")
print("First item:", merged_sorted[0])

# Save merged file
output_file = r"D:\MULTIMODAL-RAG\aic25_batch1_final.pkl"
with open(output_file, "wb") as f:
    pickle.dump(merged_sorted, f)

Final merged: 177321 items
First item: {'path': 'L21_V001/001.jpg', 'text_dense': [0.047209788113832474, 0.0019981046207249165, 0.044975679367780685, 0.031606413424015045, 0.025627221912145615, 0.042631860822439194, 0.014443962834775448, 0.007497764658182859, -0.008821159601211548, 0.04508429765701294, 0.04449836164712906, -0.03419207036495209, 0.05419206619262695, 0.04180430620908737, 0.008563888259232044, -0.04019193351268768, 0.016253652051091194, 0.022165440022945404, 0.009957698173820972, 0.02827145718038082, -0.02632327750325203, -0.012086602859199047, -0.0022614060435444117, -0.016625571995973587, -0.024870963767170906, 0.027020001783967018, -0.018216649070382118, 0.020771600306034088, -0.05414484068751335, -0.016520025208592415, -0.012924643233418465, -0.024771682918071747, 0.0007457069586962461, 0.0067398035898804665, -0.03152041137218475, -0.0006066207424737513, -0.03128012269735336, 0.004951532930135727, -0.0063448138535022736, -0.00795619934797287, 0.03827871009707451, -0.0

In [56]:
import pickle

file_path = r"D:\MULTIMODAL-RAG\aic25_batch1_florence2_ocr_padded.pkl"

with open(file_path, "rb") as f:
    data = pickle.load(f)
print(type(data))
print(len(data) if hasattr(data, "__len__") else "No length")
print(data[0] if isinstance(data, (list, tuple)) else data)
count_too_long = 0
long_paths = []

for item in data:
    if isinstance(item, dict) and "ocr" in item:
        if len(item["ocr"]) > 4096:
            count_too_long += 1
            long_paths.append(item.get("path", "UNKNOWN"))

print(f"Total entries: {len(data)}")
print(f"OCR strings > 4096 chars: {count_too_long}")

if long_paths:
    print("Paths with too-long OCR:")
    for p in long_paths[:10]:  # show only first 10 for sanity
        print("  ", p)

<class 'list'>
177321
{'path': 'L21_V001/001.jpg', 'ocr': 'htv9<pad>hd<pad>06:30:11<pad>giay'}
Total entries: 177321
OCR strings > 4096 chars: 0


In [57]:
import pickle

file_path = r"D:\MULTIMODAL-RAG\aic25_batch1_final.pkl"

with open(file_path, "rb") as f:
    data = pickle.load(f)

print(type(data))
print(len(data) if hasattr(data, "__len__") else "No length")
print(data[0] if isinstance(data, (list, tuple)) else data)

<class 'list'>
177321
{'path': 'L21_V001/001.jpg', 'text_dense': [0.047209788113832474, 0.0019981046207249165, 0.044975679367780685, 0.031606413424015045, 0.025627221912145615, 0.042631860822439194, 0.014443962834775448, 0.007497764658182859, -0.008821159601211548, 0.04508429765701294, 0.04449836164712906, -0.03419207036495209, 0.05419206619262695, 0.04180430620908737, 0.008563888259232044, -0.04019193351268768, 0.016253652051091194, 0.022165440022945404, 0.009957698173820972, 0.02827145718038082, -0.02632327750325203, -0.012086602859199047, -0.0022614060435444117, -0.016625571995973587, -0.024870963767170906, 0.027020001783967018, -0.018216649070382118, 0.020771600306034088, -0.05414484068751335, -0.016520025208592415, -0.012924643233418465, -0.024771682918071747, 0.0007457069586962461, 0.0067398035898804665, -0.03152041137218475, -0.0006066207424737513, -0.03128012269735336, 0.004951532930135727, -0.0063448138535022736, -0.00795619934797287, 0.03827871009707451, -0.019334858283400536